# 04. Інтерпретованість: Grad-CAM, brand shortcut, залишки

Мета цього ноутбука — не просто виміряти точність моделі, а чесно перевірити, *чому* вона робить свої передбачення. Головний ризик дизайну (див. README, Limitations) — **brand shortcut**: якщо модель навчилась мапити шильдик/решітку радіатора на середню ціну бренду, вона може мати непогані агреговані метрики, при цьому нічого не розуміючи про форму кузова, пропорції чи стан авто.

План перевірки:
1. Grad-CAM для голови `log_price` — куди дивиться модель на фото.
2. Кореляція pred vs true ціни *всередині* кожного бренду (усуває сам ефект "бренд -> середня ціна").
3. Залишки відносно віку авто — чи є систематичний біас за роком.
4. Метрики на hold-out спліті з повністю невідомими моделями авто — найчесніша оцінка узагальнення.

Всі метрики нижче рахуються прямо в ноутбуці, без прихованих утиліт, щоб було видно кожен крок обчислення.

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.join("..", "src"))

from car_price_vision.models.multitask import MultiTaskCarNet
from car_price_vision.interpret.gradcam import GradCAM, overlay_heatmap
from car_price_vision.data.transforms import eval_transforms
from car_price_vision.data.splits import make_splits
from car_price_vision.data.dataset import DVMCarDataset
from car_price_vision.metrics import within_brand_corr, r2 as r2_score_fn

warnings.filterwarnings("ignore")

DATA_ROOT = "/data/car-price-vision"
MANIFEST_PATH = os.path.join(DATA_ROOT, "manifest.csv")
CHECKPOINT_PATH = os.path.join(DATA_ROOT, "checkpoints", "baseline_convnext", "latest.pt")
SEED = 42
MAX_SAMPLE = 5000  # cap for scatter plots / batched inference below

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

manifest = pd.read_csv(MANIFEST_PATH)
print("Manifest rows:", len(manifest))
manifest.head()

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

target_norm = checkpoint["target_norm"]
backbone_name = checkpoint["backbone_name"]
img_size = checkpoint.get("img_size", 224)

print("Backbone:", backbone_name)
print("Image size:", img_size)
print("Checkpoint stage / epoch:", checkpoint.get("stage"), "/", checkpoint.get("epoch"))
print("target_norm:", target_norm)

model = MultiTaskCarNet(backbone_name=backbone_name, pretrained=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval().to(DEVICE)
None

In [ ]:
# Helper functions used throughout the notebook.

def load_image(path: str) -> Image.Image:
    # path is relative to DATA_ROOT, as stored in manifest["image_path"].
    full_path = os.path.join(DATA_ROOT, path)
    return Image.open(full_path).convert("RGB")


def preprocess(pil_img: Image.Image) -> torch.Tensor:
    # PIL image -> normalized (1, 3, H, W) tensor ready for the model.
    tensor = eval_transforms(img_size)(pil_img)
    return tensor.unsqueeze(0)


def destandardize(year_z: float, log_price_z: float) -> tuple:
    year = year_z * target_norm["year_std"] + target_norm["year_mean"]
    log_price = log_price_z * target_norm["log_price_std"] + target_norm["log_price_mean"]
    return year, float(np.exp(log_price))


def destandardize_batch(year_z: torch.Tensor, log_price_z: torch.Tensor):
    year = year_z.detach().cpu().numpy() * target_norm["year_std"] + target_norm["year_mean"]
    log_price = log_price_z.detach().cpu().numpy() * target_norm["log_price_std"] + target_norm["log_price_mean"]
    return year, np.exp(log_price)


def predict(pil_img: Image.Image) -> tuple:
    # Single-image inference -> (year, price_gbp), de-standardized.
    x = preprocess(pil_img).to(DEVICE)
    with torch.no_grad():
        out = model(x)
    return destandardize(out["year"].item(), out["log_price"].item())


@torch.no_grad()
def predict_batch(df_subset: pd.DataFrame, batch_size: int = 128, num_workers: int = 4) -> pd.DataFrame:
    # Batched inference over a manifest-like DataFrame; returns a copy with
    # `pred_year` / `pred_price` columns added, row order preserved.
    ds = DVMCarDataset(df_subset, data_root=DATA_ROOT, transform=eval_transforms(img_size), target_norm=None)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    pred_years, pred_prices = [], []
    for images, _ in loader:
        images = images.to(DEVICE)
        out = model(images)
        year, price = destandardize_batch(out["year"], out["log_price"])
        pred_years.append(year)
        pred_prices.append(price)

    result = df_subset.reset_index(drop=True).copy()
    result["pred_year"] = np.concatenate(pred_years)
    result["pred_price"] = np.concatenate(pred_prices)
    return result

## Підготовка даних: той самий спліт, що й на тренуванні

Щоб коректно порівняти "бачені" та "небачені моделі" (`test` vs `holdout`), потрібно відтворити точно той самий спліт, що використовувався під час тренування: спочатку `subset_size=80000` семплування маніфесту (`random_state=42`), а потім `make_splits(mode="by_model", ...)` з тим самим `seed=42`. Інакше holdout нижче не відповідатиме моделям, яких модель дійсно ніколи не бачила під час навчання.

In [ ]:
df_sub = manifest.sample(n=80000, random_state=SEED).reset_index(drop=True)

splits = make_splits(
    df_sub,
    mode="by_model",
    holdout_models_frac=0.1,
    val_frac=0.1,
    test_frac=0.1,
    seed=SEED,
)

test_df = df_sub.iloc[splits["test"]].reset_index(drop=True)
holdout_df = df_sub.iloc[splits["holdout"]].reset_index(drop=True)

print("subset:", len(df_sub), "| train:", len(splits["train"]), "| val:", len(splits["val"]),
      "| test:", len(test_df), "| holdout (unseen models):", len(holdout_df))
print("Unique models in test:", test_df["model"].nunique(), "| in holdout:", holdout_df["model"].nunique())
print("Overlap between test and holdout model names (must be 0):",
      len(set(test_df["model"]) & set(holdout_df["model"])))

## 1. Grad-CAM галерея

Обираємо кілька фото авто по всьому діапазону ціни (дешеві / середні / дорогі за `price_gbp`) і рахуємо Grad-CAM для голови `log_price`: куди модель "дивиться", коли оцінює ціну. Якщо тепло-карта стабільно концентрується на шильдику/решітці радіатора — це ознака brand shortcut. Якщо вона розподілена по кузову (лінії дверей, колісні арки, дах, загальні пропорції) — модель, схоже, реагує на форму, а не лише на логотип.

In [ ]:
n_samples = 8
quantiles = np.linspace(0.05, 0.95, n_samples)
price_sorted = df_sub.sort_values("price_gbp").reset_index(drop=True)
candidate_idx = (quantiles * (len(price_sorted) - 1)).astype(int)
candidates = price_sorted.iloc[candidate_idx]

target_layer = model.backbone.default_target_layer()
cam_extractor = GradCAM(model, target_layer)

gallery = []
for _, row in candidates.iterrows():
    try:
        pil_img = load_image(row["image_path"])
        x = preprocess(pil_img).to(DEVICE)
        cam = cam_extractor(x, output_key="log_price")[0]
        overlay = overlay_heatmap(pil_img, cam)
        pred_year, pred_price = predict(pil_img)
        gallery.append({
            "original": pil_img,
            "overlay": overlay,
            "true_price": row["price_gbp"],
            "pred_price": pred_price,
            "brand": row["brand"],
            "model": row["model"],
        })
    except Exception as exc:
        print(f"Skipping {row.get('image_path')}: {exc}")

cam_extractor.remove_hooks()
print(f"Successfully processed {len(gallery)}/{len(candidates)} images.")

if not gallery:
    print("No Grad-CAM samples succeeded -- check DATA_ROOT / image paths.")
else:
    fig, axes = plt.subplots(len(gallery), 2, figsize=(7, 3.2 * len(gallery)))
    if len(gallery) == 1:
        axes = axes.reshape(1, 2)
    for i, item in enumerate(gallery):
        axes[i, 0].imshow(item["original"])
        axes[i, 0].set_title(f"{item['brand']} {item['model']}\ntrue \u00a3{item['true_price']:,.0f}", fontsize=9)
        axes[i, 0].axis("off")
        axes[i, 1].imshow(item["overlay"])
        axes[i, 1].set_title(f"Grad-CAM (log_price)\npred \u00a3{item['pred_price']:,.0f}", fontsize=9)
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.show()

## 2. Within-brand кореляція ціни

Кореляція pred vs true ціни *по всьому* датасету майже завжди висока — просто тому, що дорогі бренди (BMW, Mercedes) в середньому дорожчі за бюджетні (Ford, Vauxhall), і модель може вивчити це без жодного розуміння фото. Набагато чесніший тест — кореляція **всередині одного бренду**: якщо вона теж додатна, модель розрізняє дешевші й дорожчі авто одного бренду за фото (стан, покоління, комплектація), а не просто зчитує шильдик.

Рахуємо на test-спліті (ті самі бренди, що модель бачила під час тренування, але конкретні фото — ні).

In [ ]:
test_sample = test_df.sample(n=min(MAX_SAMPLE, len(test_df)), random_state=SEED).reset_index(drop=True)
test_pred = predict_batch(test_sample)

top_brands = test_pred["brand"].value_counts().head(6).index.tolist()
print("Top-6 brands by count in the test sample:", top_brands)

brand_subset = test_pred[test_pred["brand"].isin(top_brands)].copy()
corr_result = within_brand_corr(brand_subset, pred_col="pred_price", true_col="price_gbp")

for brand in top_brands:
    r = corr_result["per_brand"].get(brand)
    n = int((brand_subset["brand"] == brand).sum())
    r_str = "n/a" if r is None else f"{r:.3f}"
    print(f"{brand:>15s}: n={n:5d}  Pearson r = {r_str}")
print("\nMean within-brand r (brands with >=3 samples):", round(corr_result["mean"], 3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, brand in zip(axes.flat, top_brands):
    sub = brand_subset[brand_subset["brand"] == brand]
    ax.scatter(sub["price_gbp"], sub["pred_price"], alpha=0.4, s=12)
    lims = [0, max(sub["price_gbp"].max(), sub["pred_price"].max()) * 1.05]
    ax.plot(lims, lims, "r--", linewidth=1)
    r = corr_result["per_brand"].get(brand)
    ax.set_title(f"{brand} (n={len(sub)}, r={r:.2f})" if r is not None else str(brand))
    ax.set_xlabel("true price, £")
    ax.set_ylabel("pred price, £")
plt.tight_layout()
plt.show()

bar_brands = [b for b in top_brands if corr_result["per_brand"].get(b) is not None]
bar_values = [corr_result["per_brand"][b] for b in bar_brands]
plt.figure(figsize=(8, 4))
plt.bar(bar_brands, bar_values, color="steelblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Pearson r (pred vs true price)")
plt.title("Within-brand кореляція ціни (топ-6 брендів)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 3. Залишки за віком авто

`age = advert_year - year` — вік авто на момент розміщення оголошення. Якщо модель має систематичний біас за віком (наприклад, недооцінює старі авто чи переоцінює нові через дисбаланс у навчальних даних або інфляцію цін у GBP за роками), це буде видно як тренд у залишках. Це також чесний привід обговорити потенційний **leakage через вік**: якщо модель головно вивчила "старіше -> дешевше" без урахування бренду чи стану, це інша форма "легкого" сигналу (рік -> ціна), а не справжнє розпізнавання зовнішнього вигляду з фото.

In [ ]:
age_df = test_pred.copy()
age_df["age"] = age_df["advert_year"] - age_df["year"]
age_df["log_price_true"] = np.log(age_df["price_gbp"].clip(lower=1e-6))
age_df["log_price_pred"] = np.log(age_df["pred_price"].clip(lower=1e-6))
age_df["log_price_residual"] = age_df["log_price_pred"] - age_df["log_price_true"]

age_df = age_df[(age_df["age"] >= 0) & (age_df["age"] <= 40)]

bins = np.arange(0, 41, 2)
age_df["age_bin"] = pd.cut(age_df["age"], bins=bins)
binned = age_df.groupby("age_bin", observed=True)["log_price_residual"].agg(["mean", "count"]).reset_index()
bin_centers = [interval.mid for interval in binned["age_bin"]]

trend = np.polyfit(age_df["age"], age_df["log_price_residual"], deg=1)
trend_x = np.array([age_df["age"].min(), age_df["age"].max()])
trend_y = np.polyval(trend, trend_x)

plt.figure(figsize=(9, 5))
plt.scatter(age_df["age"], age_df["log_price_residual"], alpha=0.15, s=10, label="окремі авто")
plt.plot(bin_centers, binned["mean"], "o-", color="darkorange", label="середнє по біну (2 роки)")
plt.plot(trend_x, trend_y, "r--", label=f"тренд (нахил={trend[0]:.4f}/рік)")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("вік авто на момент оголошення, років")
plt.ylabel("залишок у log-price (pred - true)")
plt.title("Залишки ціни vs вік авто (test-спліт)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Нахил тренду залишків за віком: {trend[0]:.5f} log-price / рік (позитивний = переоцінка старих авто)")

## 4. Hold-out: небачені моделі авто

Найчесніша перевірка узагальнення — метрики на `holdout`, спліті, зібраному з моделей авто, яких не було в train/val/test взагалі. Порівнюємо з test (бачені моделі, нові фото), щоб оцінити **generalization gap**: наскільки сильно якість падає, коли модель зустрічає геть новий кузов/модель, а не просто нове фото відомої моделі.

In [ ]:
holdout_sample = holdout_df.sample(n=min(MAX_SAMPLE, len(holdout_df)), random_state=SEED).reset_index(drop=True)
holdout_pred = predict_batch(holdout_sample)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    mae_years = float(np.mean(np.abs(pred_df["pred_year"] - pred_df["year"])))
    true_price = pred_df["price_gbp"].to_numpy(dtype=np.float64)
    pred_price = pred_df["pred_price"].to_numpy(dtype=np.float64)
    mape = float(np.mean(np.abs(pred_price - true_price) / np.maximum(true_price, 1e-6)) * 100.0)
    r2_price = r2_score_fn(np.log(np.maximum(pred_price, 1e-6)), np.log(np.maximum(true_price, 1e-6)))
    return {"mae_years": mae_years, "mape_pct": mape, "r2_log_price": r2_price, "n": len(pred_df)}


test_metrics = compute_metrics(test_pred)
holdout_metrics = compute_metrics(holdout_pred)

metrics_table = pd.DataFrame(
    [test_metrics, holdout_metrics],
    index=["test (бачені моделі)", "holdout (небачені моделі)"],
)
metrics_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for ax, (name, df_m) in zip(axes, [("test", test_pred), ("holdout", holdout_pred)]):
    ax.scatter(df_m["price_gbp"], df_m["pred_price"], alpha=0.15, s=10)
    lo = max(1.0, min(df_m["price_gbp"].min(), df_m["pred_price"].min()))
    hi = max(df_m["price_gbp"].max(), df_m["pred_price"].max())
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("true price, £ (log scale)")
    ax.set_ylabel("pred price, £ (log scale)")
    ax.set_title(f"{name}: n={len(df_m)}")
plt.tight_layout()
plt.show()

## Висновок

Критерії, за якими варто читати результати вище (числа з'являються після запуску на GPU-боксі):

- **Grad-CAM (розділ 1)**: якщо тепло-карти в галереї стабільно "запалюються" на шильдику/решітці радіатора незалежно від бренду й ціни — це прямий сигнал brand/logo shortcut. Якщо активація розподілена по кузову (лінії дверей, дах, колісні арки, загальні пропорції), модель, схоже, реагує на форму авто, а не лише на логотип.
- **Within-brand кореляція (розділ 2)**: середнє `r` помітно вище нуля для більшості з топ-6 брендів означає, що модель розрізняє дешевші й дорожчі авто *в межах одного бренду* — тобто вона не просто вивчила "бренд -> середня ціна". Якщо `r` близьке до нуля або від'ємне для більшості брендів, це підтверджує ризик shortcut: загальна кореляція може виглядати добре виключно за рахунок між-брендової варіації цін.
- **Залишки за віком (розділ 3)**: помітний ненульовий нахил тренду залишків означає систематичний біас за віком авто. Це окрема форма "легкого" сигналу (рік -> ціна), яку варто відрізняти від brand shortcut, хоча природа проблеми та сама — модель може економити на справжньому розпізнаванні зовнішнього вигляду.
- **Test vs holdout (розділ 4)**: розрив між метриками на test (бачені моделі) і holdout (повністю невідомі моделі авто) — найчесніша оцінка узагальнення. Великий розрив (суттєво гірший MAPE/R² на holdout) свідчить, що частина точності на test тримається на запам'ятовуванні конкретних моделей авто, а не на перенесених візуальних ознаках ціни.

Загальний висновок щодо ризику brand shortcut варто формулювати, лише поєднавши всі чотири сигнали разом: сильний shortcut дав би одночасно "гарячі" шильдики на Grad-CAM, слабку/нульову within-brand кореляцію і великий test-holdout розрив. Якщо ж within-brand кореляція додатна і Grad-CAM показує увагу до кузова, а не тільки до логотипу, — модель, найімовірніше, вивчила щось змістовніше за просте мапування бренду на середню ціну, навіть якщо певний внесок бренду в передбачення, звісно, залишається (і це очікувано: бренд справді корелює з ціною і в реальному світі).